# 03 — Consolidación de JSON históricos en Parquet analítico

**Proyecto:** RepostaPro — Optimización del repostaje en flotas comerciales  
**Autor:** Víctor González Martín  
**Notebook:** 03 — Consolidación del histórico descargado en formato Parquet

## Objetivo del notebook

Consolidar los ficheros JSON descargados en el notebook 2 (paso anterior) en un dataset analítico unificado y eficiente. La consolidación aplica las limpiezas estándar definidas en el módulo `src/consolidacion.py` (conversión de tipos, gestión de valores ausentes, normalización del campo Rótulo, etiquetado de outliers geográficos) y produce un Parquet por año, particionado para facilitar análisis selectivos.

## Estrategia adoptada

- **Particionado por año**: tres ficheros Parquet (`historico_carburantes_2024.parquet`, `2025.parquet`, `2026.parquet`).
- **Formato long**: una fila por par estación × día, con la columna `fecha` como variable temporal.
- **Limpieza aplicada durante la consolidación**: aprovechamos el único pase sobre los 10 GB de datos brutos.
- **Compresión Snappy**: equilibrio entre velocidad de lectura y tamaño de fichero.

## Importación del módulo de consolidación

El módulo `src/consolidacion.py` se importa como librería, igual que se hizo con `src/descarga.py` en el notebook anterior.

In [2]:
# Añadir la raíz del proyecto al sys.path para importar desde src/
from pathlib import Path
import sys

RAIZ_PROYECTO = Path("..").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")

# Importamos las funciones del módulo de consolidación
from src.consolidacion import consolidar_ano, cargar_y_limpiar_json

import pandas as pd
import numpy as np

print("✓ Módulo src.consolidacion importado correctamente")

# Rutas del proyecto
CARPETA_RAW = Path("../data/raw")
CARPETA_PROCESSED = Path("../data/processed")

# Verificación de que las carpetas existen
print(f"\nCarpeta raw:       {CARPETA_RAW.resolve()}")
print(f"Carpeta processed: {CARPETA_PROCESSED.resolve()}")

# Conteo de JSON disponibles
n_jsons = len(list(CARPETA_RAW.glob("precios_*.json")))
print(f"\nFicheros JSON encontrados en data/raw/: {n_jsons:,}")

Raíz del proyecto: C:\TFM
✓ Módulo src.consolidacion importado correctamente

Carpeta raw:       C:\TFM\data\raw
Carpeta processed: C:\TFM\data\processed

Ficheros JSON encontrados en data/raw/: 897


## Prueba unitaria: cargar y limpiar un único JSON

Antes de lanzar la consolidación masiva, validamos que `cargar_y_limpiar_json` funciona correctamente sobre un único fichero. Probamos con el JSON del 1 de enero de 2024 y revisamos:

- El número de filas resultante.
- Los tipos de dato de las columnas.
- Que `Rotulo_normalizado` se ha generado correctamente.
- Que la columna `fecha` está presente y bien tipada.

In [3]:
# Prueba unitaria del módulo: una sola fecha
from datetime import date

fecha_prueba = date(2024, 1, 1)
ruta_prueba = CARPETA_RAW / f"precios_{fecha_prueba.isoformat()}.json"

print(f"Probando con: {ruta_prueba.name}\n")

df_prueba = cargar_y_limpiar_json(ruta_prueba, fecha_prueba)

if df_prueba is None:
    print("Error: no se ha podido cargar el fichero.")
else:
    print(f"  DataFrame generado correctamente")
    print(f"  Filas:    {len(df_prueba):,}")
    print(f"  Columnas: {df_prueba.shape[1]}")
    print(f"  Memoria:  {df_prueba.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
    print(f"\nTipos de columna:")
    print(df_prueba.dtypes)
    print(f"\nPrimeras 3 filas (columnas principales):")
    cols_muestra = ["IDEESS", "fecha", "Rótulo", "Rotulo_normalizado","Provincia", "Precio Gasoleo A", "outlier_geografico"]
    cols_existentes = [c for c in cols_muestra if c in df_prueba.columns]
    print(df_prueba[cols_existentes].head(3).to_string())

Probando con: precios_2024-01-01.json

  DataFrame generado correctamente
  Filas:    10,943
  Columnas: 25
  Memoria:  3.29 MB

Tipos de columna:
IDEESS                                          str
IDMunicipio                                     str
IDProvincia                                     str
IDCCAA                                          str
C.P.                                            str
Dirección                                       str
Localidad                                       str
Municipio                                       str
Provincia                                       str
Latitud                                     float64
Longitud (WGS84)                            float64
Rótulo                                          str
Horario                                         str
Tipo Venta                                      str
Precio Gasoleo A                            float64
Precio Gasoleo Premium                      float64
Precio Gasolina 95 E5

## Validación de la normalización de marcas

Antes de consolidar, verificamos que la función `normalizar_rotulo` del módulo replica correctamente el comportamiento. Esperamos ver:

- Cepsa y Moeve agrupados bajo `CEPSA-MOEVE`.
- Repsol con número alto de estaciones.
- BonÀrea, Plenergy y Ballenoil como marcas separadas.

In [4]:
# Verificación del ranking de marcas tras la normalización
ranking_marcas = df_prueba["Rotulo_normalizado"].value_counts()

print("TOP 15 MARCAS NORMALIZADAS (snapshot 01/01/2024)")
print("=" * 60)
print(ranking_marcas.head(15).to_string())

# Cobertura de la normalización
n_normalizadas = ranking_marcas.head(20).sum()  # las 20 más grandes
n_total = len(df_prueba)
print(f"\nCobertura del ranking TOP 20: {n_normalizadas:,} de {n_total:,} estaciones "
      f"({n_normalizadas/n_total*100:.1f}% del mercado)")

TOP 15 MARCAS NORMALIZADAS (snapshot 01/01/2024)
Rotulo_normalizado
REPSOL         2781
CEPSA-MOEVE    1244
BP              640
GALP            481
SHELL           387
BALLENOIL       325
PLENERGY        263
PETRONOR        170
PETROPRIX       151
CARREFOUR       145
DISA            140
AVIA            112
Q8               91
CAMPSA           73
BONAREA          63

Cobertura del ranking TOP 20: 7,328 de 10,943 estaciones (67.0% del mercado)


## Consolidación masiva por año

Si las pruebas anteriores han sido exitosas, lanzamos la consolidación completa de los tres años. Cada año produce un fichero Parquet independiente en `data/processed/`.

**Importante**: el proceso requiere ~3-5 GB de RAM por año durante la consolidación. Si el ordenador tiene poca memoria libre, puede ser conveniente cerrar otras aplicaciones antes de continuar. Duración estimada: 3-8 minutos por año.

In [5]:
# CONSOLIDACIÓN MASIVA POR AÑO

print("=" * 70)
print("LANZANDO CONSOLIDACIÓN COMPLETA")
print("=" * 70)

resultados = {}

for ano in [2024, 2025, 2026]:
    print(f"\n>>> Procesando año {ano}...\n")
    resultado = consolidar_ano(ano=ano,carpeta_raw=CARPETA_RAW,carpeta_processed=CARPETA_PROCESSED,)
    if resultado is not None:
        resultados[ano] = resultado

print("\n" + "=" * 70)
print("CONSOLIDACIÓN COMPLETA")
print("=" * 70)

2026-06-22 21:31:43 [INFO] ======================================================================
2026-06-22 21:31:43 [INFO] CONSOLIDANDO AÑO 2024
2026-06-22 21:31:43 [INFO] Ficheros a procesar: 366
2026-06-22 21:31:43 [INFO] ======================================================================


LANZANDO CONSOLIDACIÓN COMPLETA

>>> Procesando año 2024...



2026-06-22 21:31:51 [INFO]   [30/366] Procesados hasta 2024-01-30
2026-06-22 21:32:00 [INFO]   [60/366] Procesados hasta 2024-02-29
2026-06-22 21:32:09 [INFO]   [90/366] Procesados hasta 2024-03-30
2026-06-22 21:32:18 [INFO]   [120/366] Procesados hasta 2024-04-29
2026-06-22 21:32:27 [INFO]   [150/366] Procesados hasta 2024-05-29
2026-06-22 21:32:36 [INFO]   [180/366] Procesados hasta 2024-06-28
2026-06-22 21:32:45 [INFO]   [210/366] Procesados hasta 2024-07-28
2026-06-22 21:32:54 [INFO]   [240/366] Procesados hasta 2024-08-27
2026-06-22 21:33:04 [INFO]   [270/366] Procesados hasta 2024-09-26
2026-06-22 21:33:14 [INFO]   [300/366] Procesados hasta 2024-10-26
2026-06-22 21:33:23 [INFO]   [330/366] Procesados hasta 2024-11-25
2026-06-22 21:33:29 [WARNING]   [349/366] OMITIDO (fichero corrupto): 2024-12-14
2026-06-22 21:33:33 [INFO]   [360/366] Procesados hasta 2024-12-25
2026-06-22 21:33:35 [INFO]   [366/366] Procesados hasta 2024-12-31
2026-06-22 21:33:39 [INFO] ========================


>>> Procesando año 2025...



2026-06-22 21:33:47 [INFO]   [30/365] Procesados hasta 2025-01-30
2026-06-22 21:33:56 [INFO]   [60/365] Procesados hasta 2025-03-01
2026-06-22 21:34:05 [INFO]   [90/365] Procesados hasta 2025-03-31
2026-06-22 21:34:14 [INFO]   [120/365] Procesados hasta 2025-04-30
2026-06-22 21:34:20 [WARNING]   [141/365] OMITIDO (fichero corrupto): 2025-05-21
2026-06-22 21:34:20 [WARNING]   [142/365] OMITIDO (fichero corrupto): 2025-05-22
2026-06-22 21:34:23 [INFO]   [150/365] Procesados hasta 2025-05-30
2026-06-22 21:34:32 [INFO]   [180/365] Procesados hasta 2025-06-29
2026-06-22 21:34:42 [INFO]   [210/365] Procesados hasta 2025-07-29
2026-06-22 21:34:51 [INFO]   [240/365] Procesados hasta 2025-08-28
2026-06-22 21:35:00 [INFO]   [270/365] Procesados hasta 2025-09-27
2026-06-22 21:35:09 [INFO]   [300/365] Procesados hasta 2025-10-27
2026-06-22 21:35:19 [INFO]   [330/365] Procesados hasta 2025-11-26
2026-06-22 21:35:28 [INFO]   [360/365] Procesados hasta 2025-12-26
2026-06-22 21:35:30 [INFO]   [365/365


>>> Procesando año 2026...



2026-06-22 21:35:43 [INFO]   [30/165] Procesados hasta 2026-01-30
2026-06-22 21:35:51 [INFO]   [60/165] Procesados hasta 2026-03-01
2026-06-22 21:36:00 [INFO]   [90/165] Procesados hasta 2026-03-31
2026-06-22 21:36:10 [INFO]   [120/165] Procesados hasta 2026-04-30
2026-06-22 21:36:19 [INFO]   [150/165] Procesados hasta 2026-05-30
2026-06-22 21:36:24 [INFO]   [165/165] Procesados hasta 2026-06-14
2026-06-22 21:36:26 [INFO] ======================================================================
2026-06-22 21:36:26 [INFO] AÑO 2026 CONSOLIDADO
2026-06-22 21:36:26 [INFO] Filas:      1,879,801
2026-06-22 21:36:26 [INFO] Columnas:   25
2026-06-22 21:36:26 [INFO] Omitidos:   0 ficheros
2026-06-22 21:36:26 [INFO] Duración:   0:00:51.981251
2026-06-22 21:36:26 [INFO] Tamaño:     35.10 MB
2026-06-22 21:36:26 [INFO] Destino:    ..\data\processed\historico_carburantes_2026.parquet
2026-06-22 21:36:26 [INFO] ======================================================================



CONSOLIDACIÓN COMPLETA


## Resumen de la consolidación

Inspeccionamos el resultado de los tres años: número de filas, tamaño en disco, ratio de compresión respecto a los JSON originales.

In [6]:
# Resumen consolidado
print(f"{'Año':<6} {'Días':<8} {'Filas':<15} {'Columnas':<10} {'Tamaño (MB)':<15}")
print("-" * 60)

total_filas = 0
total_mb = 0

for ano, res in sorted(resultados.items()):
    print(f"{ano:<6} {res['n_ficheros']:<8} {res['n_filas']:>13,}  "
          f"{res['n_columnas']:<10} {res['tamano_mb']:>10.2f}")
    total_filas += res["n_filas"]
    total_mb += res["tamano_mb"]

print("-" * 60)
print(f"{'TOTAL':<6} {'':<8} {total_filas:>13,}  {'':<10} {total_mb:>10.2f}")

# Comparativa de tamaño con los JSON originales
tamano_json_total_mb = sum(
    f.stat().st_size for f in CARPETA_RAW.glob("precios_*.json")) / 1024 / 1024

print(f"\nTamaño total JSON original:    {tamano_json_total_mb:,.0f} MB")
print(f"Tamaño total Parquet final:    {total_mb:,.0f} MB")
print(f"Ratio de compresión:           {tamano_json_total_mb / total_mb:.1f}x más pequeño")

Año    Días     Filas           Columnas   Tamaño (MB)    
------------------------------------------------------------
2024   366          4,079,778  25              75.52
2025   365          4,080,542  25              76.26
2026   165          1,879,801  25              35.10
------------------------------------------------------------
TOTAL              10,040,121                 186.89

Tamaño total JSON original:    11,096 MB
Tamaño total Parquet final:    187 MB
Ratio de compresión:           59.4x más pequeño


## Validación final del Parquet consolidado

Para verificar que los Parquet generados son correctos y reutilizables, cargamos uno de ellos y comprobamos la integridad de los datos.

In [13]:
# Validación final: cargar un Parquet y comprobar su contenido
ano_validacion = 2024
ruta_parquet = CARPETA_PROCESSED / f"historico_carburantes_{ano_validacion}.parquet"

print(f"Cargando Parquet de validación: {ruta_parquet.name}")
df_validacion = pd.read_parquet(ruta_parquet)

print(f"\nCargado correctamente")
print(f"  Filas:    {len(df_validacion):,}")
print(f"  Columnas: {df_validacion.shape[1]}")
print(f"  Memoria:  {df_validacion.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

print(f"\nRango de fechas presente:")
print(f"  Fecha mínima: {df_validacion['fecha'].min()}")
print(f"  Fecha máxima: {df_validacion['fecha'].max()}")
print(f"  Días únicos:  {df_validacion['fecha'].nunique()}")

print(f"\nEstaciones únicas en {ano_validacion}: {df_validacion['IDEESS'].nunique():,}")
print(f"Marcas normalizadas únicas:           {df_validacion['Rotulo_normalizado'].nunique():,}")
print(f"Outliers geográficos detectados:      {df_validacion['outlier_geografico'].sum():,}")

# Estadísticas mínimas para verificar la integridad
print(f"\nPrecio medio Gasóleo A en {ano_validacion}: {df_validacion['Precio Gasoleo A'].mean():.3f} €/L")
print(f"Precio medio Gasolina 95 en {ano_validacion}: {df_validacion['Precio Gasolina 95 E5'].mean():.3f} €/L")

Cargando Parquet de validación: historico_carburantes_2024.parquet

Cargado correctamente
  Filas:    4,079,778
  Columnas: 25
  Memoria:  1228.07 MB

Rango de fechas presente:
  Fecha mínima: 2024-01-01 00:00:00
  Fecha máxima: 2024-12-31 00:00:00
  Días únicos:  365

Estaciones únicas en 2024: 11,584
Marcas normalizadas únicas:           2,780
Outliers geográficos detectados:      712

Precio medio Gasóleo A en 2024: 1.468 €/L
Precio medio Gasolina 95 en 2024: 1.572 €/L
